# Additional Keyframe Extractor (Kaggle)

Notebook độc lập này đọc export CSV, tạo JPEG additional keyframe lên Cloudflare R2 và sinh SQL `INSERT` cho PostgreSQL. Không kết nối hay thực thi SQL trên database.

**Kaggle dependencies:** `boto3`, `numpy`, `Pillow`, `torch`, `transformers` và `ffmpeg/ffprobe`. Bật Internet để Kaggle tải SigLIP2 lần đầu. Nếu image Kaggle thiếu package, notebook chỉ cài package thiếu; không ghi secret vào notebook.


In [ ]:
# Configuration — edit only this cell for a Kaggle run.
from __future__ import annotations
import os
from pathlib import Path

INPUT_DIR = Path(os.environ.get('KF_INPUT_DIR', '/kaggle/working/btc-keyframe-export'))
INPUT_DIR.mkdir(parents=True, exist_ok=True)
if not (INPUT_DIR / 'videos.csv').is_file():
    video_rows = [('L21_V001','https://pub-c8f3587e831a418ebf0d427203860188.r2.dev/data2/videos/L21_V001.mp4'),('L21_V002','https://pub-c8f3587e831a418ebf0d427203860188.r2.dev/data2/videos/L21_V002.mp4'),('L21_V003','https://pub-c8f3587e831a418ebf0d427203860188.r2.dev/data2/videos/L21_V003.mp4'),('L21_V005','https://pub-c8f3587e831a418ebf0d427203860188.r2.dev/data2/videos/L21_V005.mp4'),('L21_V006','https://pub-c8f3587e831a418ebf0d427203860188.r2.dev/data2/videos/L21_V006.mp4'),('L21_V007','https://pub-c8f3587e831a418ebf0d427203860188.r2.dev/data2/videos/L21_V007.mp4'),('L21_V008','https://pub-c8f3587e831a418ebf0d427203860188.r2.dev/data2/videos/L21_V008.mp4'),('L21_V009','https://pub-c8f3587e831a418ebf0d427203860188.r2.dev/data2/videos/L21_V009.mp4'),('L21_V010','https://pub-c8f3587e831a418ebf0d427203860188.r2.dev/data2/videos/L21_V010.mp4'),('L21_V011','https://pub-c8f3587e831a418ebf0d427203860188.r2.dev/data2/videos/L21_V011.mp4')]
    (INPUT_DIR / 'videos.csv').write_text('video_id,video_url\n' + '\n'.join(f'{video_id},{url}' for video_id,url in video_rows) + '\n', encoding='utf-8')
    import base64, gzip
    embedded = {'shot.csv':'...','keyframe.csv':'...'}
    for name, payload in embedded.items(): (INPUT_DIR / name).write_bytes(gzip.decompress(base64.b64decode(payload)))
    # Exact 326-shot output produced by the Shot Extractor test for L21_V001.
    (INPUT_DIR / 'shot.csv').write_bytes(gzip.decompress(base64.b64decode('H4sIAAAAAAAEAGWby6otyY2G5w1+kzVIKe7v0LMGTwtDVWMPbIOraPz4/n+Fcm1JzYFcocOnnXFRRCgUyt//+s8/fvnbr5//+9uvv/2Thd/tP/7x62///vz+x1/+9ccvf//989s/fuXPlf/3X3/5+28g/23//ZX+9F//rfLLn59Hfvmf53k+r/R58E/2w5/RMiU/lFxGpLf2Gf3TumZWf1h1TNqCSuvt0+XJdPuhX06GzgVSPn2sTPcfujsni43oY3/GczI9fujhnOzNaguaOHum5w89ndOnPyTHZ4pkev3QyznVBzpT9DNnaeX+obdz2h/Ufk7oa/nb54c+zungG5bqZ81MSxhBlC+oe0BpTf2gkPE4lOJge9jve6zP0VxzCaOJ8gWnHjyP0hR2+fNhPFG+5FybA7U3e6qNrBCGFOWLLrEeetr8qLRSozCqKF90Nb4HLFqkUqoUBhbli6Jr2KsqfOxsZRLGVpaj64gpbIxfW9l0JAwvyhfd1ybaGhjCYj0SRhjli+5hJtfRzTrKGzQMsr7oeaZZBt+wJNu+hmFG+aKnsXN1KQ1E8jhonLbq6BnKKm10m1ZD0jDSKF/07GkKsCQ9Y2eFMNIoX1SeLeymM86naVlKNAw1ys6KyORqolhOWut57DQMNsrOijRTaX1jFXqKShhulJ0VOZ0zo2PuNnR3VgkDjrKzog+bBBqr4pTS+jDkKDsrqlaxCStoc+WKtTDoKDsreLJimCmftnY2rBaGvcnLis/ytTGD6zi2MPAoO4uZZG/hSLbTylviot1eVlq3txz8lY6pm1XC4KPsrDTrONBc70sntzD6KDuL/YS9ALp/eitD2cLoo+ys9MW+Br0/vWselxZGv62XlcHtgvTB1jLyCtTC6KPsrIzJGQAae9Msy2ILo4+ys4Jdhc2fWBj7bnnK9zD6KDsrs3OtAD0+vY5LD6Pf5WVlTusxG5dzco/1MPooOyvzcNsGzV1151nZw+j39rKybFcDze217H897tv9ZWVP8yHQtM8Yp7wljD7KzupzGnfkcbi3rDyUPYw+ys4qLJobra7zmbDnrBJGH2VnsSvQxQE9P3O0bJY9jD7KzqoqrRM09rC5ikoYfZSdVV0cHtBQ2cUHGGH0x/Oy2oQrIGj4GBjPrBJGf8jLarO9AjR2yadsKyOMPsrOYs9a3PGexe21viWM/viycFIe27f5Fn1KW8Loo+wslqbr0sBhxDJWVKLrNl5WZ2eTQEPlaKlYGH2UncXWTbsB3bDNVpUw+tyxLquYjtyOH1MpljzC6KPsLLbfZls+LHlLLyph9FF2Vtdgx4GGCrbkpDLD6M/nZWHvXJtA49HK7jrD6E95WWzdNGjQ8DKwNGWVMPooOwtviL4C6PnZ8N+yShj92V5Wj9C3xhvxWFUljD7KzuqZnAN7UWWfPF9mGH2UndVDVdL9s0/tsei9z5dt8IP4lrPMW8rGP8Pomw9sbMOfur7V+Bw5pZPD6KPsbHsOlwDQB05xbUsYfZSdbbAB+leKtpxezi0rjP76sk1saT4wazzKMr7C6C952aabHQd6w90qZ7oVRh9lZ1t7uvlxmKjow9zJK4z+ai/bmthx4MDfP6d4FyuMPsrOtra5AYDGjvbUCbPC8KPsMD0xcxg5YwRTOY/mCgaAstM4ch7umsDpP7bilK14ilsvDd/Etlrgh89R3hSMgD7YpeGIsmXE+aZe/J8VzABlp1s/5jkBt+fKq+AOhrC/dMOx1zqiY9gEi1hu0w6msOWlUSvznYGzTXPkkd3BGOgkXrqNed+EbY3PMht2MIfdXrpNtWM5cL4JS0pWCgZhfrLRbcLJegyHQ/WcMlV3sAiUnca0mPamM9nl5+RptINFoOx0m9eNtFUIz+J07mAR+0u31eQqocpCdz8rxeP9fum2rpMHHNWjZ5GVgkWg7DRs/VYMSyuPIqVNJ1gEyk43mLpVz9w9Qd9kpWARR166HV2OMxIALyYrBYtA2Wn64PYm7MF47laqFywCZacxqcyM4LLzfbuY0QkWgbLT7VgEhvjhAatYxAkWgbLTaP9tk1kEPIvSpmARKDvd78JPXOxcVt4ULOKsl+aRxQ5jj71JSljrBItA2emO2WdKsAk+V1GKcZ/z0jiujqu0qFTWPXlS+OeLY0+xWai28MF9XFkrRoHsgHE85na7Au4Dnq2Vd8VgEATHuzSzWfB8V9sna8WQ0NNevNNOTevw2XvRinEhCI53uecz7YNaI5/mJIb8KDje9bEDB/hjh+Ina8UIEQTHu+rjB247So9SwxgmguB4V7cMbMF4LinvirEiHkgv3m/cjjzfVYJ8EkOCFBzvcBxtvBjpw5IheZRTaFC+OI7A+0YFhKO858haKUIoL47j87XDDf9K4MZl20iBQgiO96Y3AnGEtnFKu1K4UNqLw624fXjYrvaUPkwxQ+kvjk3UtuDGIKfQbcpa0TYgON5HbzfgwT24zVbeFW0DguMd266FPibDLFjxS29E25Av3uftDfAMaOzaG9E2IDje57ZZCR690XGazlrRNiA43tdjcfrOozR8Jk0bncSIIgXH+7o9Dx47XdfS8zGsSMHxvu3kRJ5R3JnPWaIpiKwvDnfGfBLw1F1P0Yq2oV+841RvcaD1mNbJfRijjBQc77C8Gwo67MO9y7uibeh48X4slEue7zr5+Cwx2EjB8eEbCnj0/CgbisR4IwXHx9NtVg7bUYb0PCtjyJGC46DVgk/oRT530Yq2AcHxgbG2d+HEx2uUnvswBh4pOD7caQKPPhwtH/Qkxh4pOI6f267GGOZo+RAqMfxIwfHBo6VpDdYQlp+10jVDe3HSFr3DUfQjE+e1rBVtA4LjwyOLk4c1mSUOKTEOScFxrAP9xvDoE2LZKDWMtgHB8dGnBT7BUxfOTtaKttHWi8PdtVEGT93xlD6MtgHB8TH6bRdGnZG9VtoVbQOC42PedR4827XLOh/DkhQcH7PfGm6u8/NormGMTFJwHCeGfcONjNVjJck1jMFJCo6PuW2dX7wPxHPmPozxSQqOj9XMB1hm+Qvn0KyV7qH6i497+0Oel0HlqkhilJKC41jtp9WQ10WCxb5oRduA4PjYzXoDPLWGlN6ItgHBcdj6bReWAD53aVe0DQiOz8dikOTtQneVd0XbgOA4Zsi9ROW5XuBu5BUgBi0pOD5xvDQt6OGpZWWLcUsKjmMBNjsEz+vYVuwwhi4pOD6xNppWox2ih3K7YvSSguNT7ioKnu3qZRWNAUwKjk8V8wHAs4ZzlBqmi8rx4lO77bDgWcOVgx8Sw5gUHJ/us4G/t66lhtE2xnrx2dS8ym0+2z45viAxmEnB8dnGDc8fBhjOM0oNo22M8+LTva/DQDRmS/G+YkiTguOzPzZTjnlfR4sHG6OaFByffdkOCx4z5fRiGzGwScHxOSyITh49f8q1u8TYJgXHJ5fPZTx744yiFW1j9hefS8x6wetHHyk+W4xwUnB8LgvCkR92VZ6td6ab7PnijEHPe7Xe8SypChLjnPKDT3SDaTFhQTHnSg2jbUBwfJ5pQWLwrOEpO1GMdlJwHK6dxbzB80b/yQdfiQFPCo4vaXYdIQxvqtRRjjFPCo6ve7NKHjWUcg8rMexJwfGldiglf/gsZ9gY+aTgONZ3uVqHz53vryUGPyk4vvxmBjzrWe5xJIY/KTiOI4ZdxgtvcnDU1Dy/YvyTguMLHWl3TbzmVnRleVdKdVgvvppnO/C8ziSJ0vPRNiA4jqXJ2qWstarWdkXbgOD4wvnSWqT3ufO7YgyUguOr93Gv3TbfVSMVMQhKwfHV503UYaQCz3J2iFFQCo6vvp+rhbODas9BL4lhUAqOr35uDXu3Z21XtA0IjmMX16tl7Ro5W0ZiIJSC42vYQZH85LOcK2MklILj+Lk9P3j7o6PWMNrG/uJrTMu1Ac8aYi3JWikXZr/4wmprWutZvJos/nwMhlJwfK07yo3rMeypjHKMhlJwHEdtS+4Bjxq2clciMRxKwXH4YGaBjbclvAnN7YrxUAqOr20xefJ8zrLDxoAoBcfXafc6dy7euu6naEXbgOD4whZkNdwPtY6UGkbbgOD4Oue+6zDXqZ1d3hVtA4Lj+zG3g7wyJ6l4KTEoSsHx/exjt8H08XFKLOevGBWl4DgWRbMN8Ex3qyepk5KlzovjYGK20XmSwiTLtqFPypj64vBBbw07baOPXEONcVG1C5RbQ7VLFPKs4VxFK+ZOQXAcW+RN9JuLWiv7URrjohQc3349Dp5ZT+UyXWNclILju5+bDMb36+h5/9IYF6Xg+B5ivTE6968xRmlXTKaC4PgezbwU8GgXDoAta8V8queL73EzE8F3PmdpV0ypguD4HnaTSZ7tmnkua4yLUnB8j+3v4lweK/tsGuOiFBzHyWjMm1DHd+08v1RSSp28ONYBW0XBU3fnVVRjXJSC4/CK5tXalt6Q49ga46IUHN/Lcm/Jo+enZP9QY1yUguP7Hi/Jo+dnOY1qjItScHxjqbJsiGZ5DjP7bBrjohQc335XD54JeOVmX2NclILj+9hxhTxrePLpRmNclILj2Fdubxze+NNpzFrRNiA4jmra7rCYkIhnsaiUaalfnNkD7I3Fqx1mW2rWirZx8y3tGv25/vzi2q9Ly3jlnEt98SM3QRg8a9hyCoqmxEtmXl4cy/utYbP0j6GlhtE2mH558aPXqwTP58pepaYMTAiOH73ps+DtuUsfRtuA4DivEK8Wk2hrzo+mPEz94ufmHKtl/egqCcqaUjEhOA4n2Va2xSxlnPHzKUBTNiYEx0+7aVabGbpw1vMJUWNclILjp91k8c3YNpNl8woQ46IUHMc2a9a7ef5TbBnZNmJclILjp9vdMvnO5ynvSkm57cVP7ze1Rw/f1XIERmNclILjpy9bN8Az/7drHuUYF6XgONxYs3nwm5nAUtoVbQOC42eKnVM2o1iKRTFbb4yLUnAcC4WdK8Gz/0+ZKTEuSsFxnknb5ZmL+5RdL8ZFKTh+1k2POkzo4nVW7sMYF6Xg+Fl3Vh7GchRHvdyuGBel4PhZdz0ET91W1sMYF6Xg+Nm358HzXb30fIyLUnAcDp7ZPKMofGYPVnvK2u4vfna38TpMQoFV9dKuaBsQHD/Yae1dgyffM1apYbQNCI6fvfRqLdZwSumNaBt9vTj3CNOazF7H0GfrjXFRCo6ffZNGwW8+i6cX46IUHMePrQDHPL0zy6yMcVEKjjNHzfpwclaelbN5NcZFKTh+jpqXAp49v1ZesWNclILj59hFHXnq7nyvpzEuSsFx+Nl3uYECc9Cf2rBoHBBeHr/TckweJmq0pwTbdaTE/vHl8bvu9yg8mOEgULajGByl8PJ2IjA9bhw4EBSnOYZHdfzw+L0fBkCDH7eUTwM0BkgpvDyT0Ja9zz4PeHox/xgipfDy+LXsDGowb3DkoJvGICmFl5dH5eoNZX/OXvSiqUz58sx7u3mQs5tePnZrDJRSeHkmpFnOHRMcP020jF8MlVJ4efzaTSE1mIXY88WixmAphZdn7N7S9aCBcZD/1y/RXizZa3gy4Li5TGL9wm89sl76GmR+efzeBElhageexQmMIVMNPCageRfUwLjLLs5IDJqqZbJdHr/Nxg8afN95Svuivczz5ZlEp6aHXefTcJrKS18MnFJ4eXmW5aJQY+IpOXVFY+iUwsvjd9j7lCF6bFZlk4vBU7Xsw8vj1yLs1OD7ymWNxvAphZcXXiybHq9rGnbAUs9oL7xycp55eDbu2P1Yz5GTNjSGUCm8PH5tf6UGv3YpiQoag6gUXl6e81h2njJVodWgl670CdH68vi1yE2zsFfTEujRGEi1lCvn8Wu3CNSw1NxygI2hVLWEzsvjdzV7HxwoPE9xe2MwlcLLM3nS0jChwe+wTk4X0xhOpfDyDODuq8dMWCyi2V5iQJXCy+NXrZ6NH2zhWcIwMaRK4eXxa9+lUeMwmbi4AjGoqpaqeXkRfqP0mAbf18rxLYZVKbw8fu0MQg0mOo9yZImBVQovj99tad7QUD5ndnRiaPUmel5e3k/IoMFk6VnHL313tr+8pYva++xDMrQ5z4cYXqXw8mym7WNMRcLzFLc7BlgpvDx+7dMialhmdnFdYoiVwsvz2zr/qg7OC54lPBCDrBReHr/m4VKDXxxLcYhjmJXCywsvjSwNnNdlrZcPDDUGWim8PH7V+qXbR4Y4yRS9aC+W3Xt5/N71k5cUTCcv62cMtlJ4efzum0TOdBf0VdnfY7iVwsvD/uXqde7vfZQgaAy4Unh5/M7bn6NRbz2lnuljxfPl8btt3kKD9Vw92Wd70heLz5fHS+16hxpMft85ZaPFsCuFl8evndhav7UtB7wWA68UXp6NtPnHPK6P3U9nvfgB49O+PH637UeDHyzimffNFoOvFF4e077dLHt+utMwpqVf4qeMz/jymPZi/TIYvGyjfDXQYgCWwsvj12LF1GBtyxeKLYZgKbw8fu1zLWqwX2a+omsxCEvh5YUei73vtnJJ0YufNz7ny+O3ddewD2NzaL/FQCyFl8evfa1PDeqd/IVMk/SZq3x5/F5/aeCsxGf2l1oMxlJ4eV6i2Zcf0OCXAiUc22I4lsLL4/eunxaQxXOU9kV7YXqO86JieQzU4BcTLac9tBiStUQW5/F7+xMa2G9nr/0Z7UXml2fy+P02o7M/56jti/Yi68sLby+tfcPat8o8ioHZZsnEl8fvue1bnEdz79K+aC/8yMZ5GPf97BgabF/5wLXF4CyFl8ev2vjxo0g+d25fDM82y/K/PCeT7ZvQ4CcYJdG4afo2Wr+88Ct99udiqnFbWuZtDNFSeHmmrdv4LQaxGhaBPH4xSEvh5QWOuK2AXDb4qUix6ximbZbwf3nBYdDGHRqT339zvf4PnNQjsElEAAA=')))
WORK_DIR = Path(os.environ.get('KF_WORK_DIR', '/kaggle/working/keyframe_extractor'))
VIDEO_FILE = os.environ.get('KF_VIDEOS_FILE', str(INPUT_DIR / 'videos.csv'))
SHOT_FILE = os.environ.get('KF_SHOT_FILE', str(INPUT_DIR / 'shot.csv'))
KEYFRAME_FILE = os.environ.get('KF_KEYFRAME_FILE', str(INPUT_DIR / 'keyframe.csv'))
VIDEO_START = int(os.environ.get('VIDEO_START', '0'))
VIDEO_END_RAW = os.environ.get('VIDEO_END', '')
VIDEO_END = int(VIDEO_END_RAW) if VIDEO_END_RAW else None  # half-open [start, end)
ALLOW_CHECKPOINT_MISMATCH = os.environ.get('ALLOW_CHECKPOINT_MISMATCH', '0') == '1'

TARGET_INTERVAL_MS = 2500
MIN_FRAME_GAP = 5
MAX_ADDITIONAL_PER_SHOT = 5
TRANSITION_MARGIN_FRAMES = 2
MAX_CANDIDATES = 64
MAX_REFERENCES = 16
IMAGE_BATCH_SIZE = int(os.environ.get('IMAGE_BATCH_SIZE', '128'))
SIGLIP_MODEL_ID = os.environ.get('SIGLIP_MODEL_ID', 'google/siglip2-base-patch16-224')
SIGLIP_MODEL_REVISION = os.environ.get('SIGLIP_MODEL_REVISION') or None
DOWNLOAD_WORKERS, UPLOAD_WORKERS = 2, 8
FFMPEG_EXPORT_CHUNK_SIZE = 100
SEED = 20260823

# Secrets are read only from Kaggle Secrets/environment. Never print these values.
R2_ENDPOINT_URL = os.environ.get('R2_ENDPOINT_URL')
R2_BUCKET = os.environ.get('R2_BUCKET')
R2_ACCESS_KEY_ID = os.environ.get('R2_ACCESS_KEY_ID')
R2_SECRET_ACCESS_KEY = os.environ.get('R2_SECRET_ACCESS_KEY')
R2_KEY_PREFIX = os.environ.get('R2_KEY_PREFIX', 'data2/keyframes').strip('/')

WORK_DIR.mkdir(parents=True, exist_ok=True)
assert 0 <= VIDEO_START and (VIDEO_END is None or VIDEO_END >= VIDEO_START)


In [ ]:
# Imports, reproducibility, errors and utility functions.
import csv, hashlib, io, json, math, random, re, shutil, subprocess, time, traceback
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import asdict, dataclass
from typing import Any, Iterable

import numpy as np
from PIL import Image

random.seed(SEED); np.random.seed(SEED)
try:
    import torch
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
        torch.backends.cudnn.benchmark = True
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
    GPU_NAME = torch.cuda.get_device_name(0) if DEVICE == 'cuda' else None
except ImportError as exc:
    raise RuntimeError('Missing dependency: torch. Install it in Kaggle before running.') from exc

ERRORS_PATH, REPORT_PATH = WORK_DIR / 'errors.jsonl', WORK_DIR / 'report.jsonl'
def jsonl(path: Path, row: dict[str, Any]) -> None:
    with path.open('a', encoding='utf-8') as handle: handle.write(json.dumps(row, ensure_ascii=False, default=str) + '\n')
def error(stage: str, message: str, **context: Any) -> None:
    jsonl(ERRORS_PATH, {'stage': stage, 'message': message, **context})
def run(command: list[str]) -> subprocess.CompletedProcess[str]:
    return subprocess.run(command, check=True, capture_output=True, text=True)
def even(values: list[int], count: int) -> list[int]:
    if count >= len(values): return values
    if count == 1: return [values[len(values)//2]]
    return [values[round(i*(len(values)-1)/(count-1))] for i in range(count)]
def qsql(value: Any) -> str:
    if value is None or (isinstance(value, float) and math.isnan(value)): return 'NULL'
    if isinstance(value, bool): return 'TRUE' if value else 'FALSE'
    if isinstance(value, (int, float)): return str(value)
    return "'" + str(value).replace("'", "''") + "'"
def sha256_paths(paths: Iterable[Path], extra: dict[str, Any]) -> str:
    digest = hashlib.sha256(json.dumps(extra, sort_keys=True).encode())
    for path in paths:
        digest.update(path.name.encode()); digest.update(path.read_bytes())
    return digest.hexdigest()


In [ ]:
# Input parsing and validation. Invalid rows are written to errors.jsonl and excluded.
REQUIRED_SHOTS = {'shot_id','video_id','shot_index','start_ms','end_ms','start_frame_idx','end_frame_idx'}
REQUIRED_FRAMES = {'frame_id','video_id','shot_id','timestamp_ms','fps','frame_idx','source','n','pts_time','frame_path','width','height'}

def read_rows(path: str, kind: str) -> list[dict[str, str]]:
    source = Path(path)
    if kind == 'videos' and source.suffix.lower() == '.txt':
        rows = []
        for line_no, line in enumerate(source.read_text(encoding='utf-8').splitlines(), 1):
            if not line.strip(): continue
            parts = [x.strip() for x in line.split(',', 1)]
            if len(parts) != 2: error('input', 'expected video_id,url', file=str(source), line=line_no); continue
            rows.append({'video_id': parts[0], 'video_url': parts[1]})
        return rows
    with source.open(encoding='utf-8-sig', newline='') as handle: return list(csv.DictReader(handle))

def require_columns(rows: list[dict[str,str]], needed: set[str], name: str) -> None:
    found = set(rows[0]) if rows else set()
    missing = needed - found
    if missing: raise ValueError(f'{name} missing required columns: {sorted(missing)}')

def integer(row: dict[str,str], field: str) -> int: return int(str(row[field]).strip())
def decimal(row: dict[str,str], field: str) -> float: return float(str(row[field]).strip())

def load_and_validate() -> tuple[dict[str,str], dict[str,list[dict[str,Any]]], list[dict[str,Any]]]:
    videos_raw, shots_raw, frames_raw = read_rows(VIDEO_FILE, 'videos'), read_rows(SHOT_FILE, 'shots'), read_rows(KEYFRAME_FILE, 'frames')
    require_columns(videos_raw, {'video_id','video_url'}, 'videos')
    require_columns(shots_raw, REQUIRED_SHOTS, 'shot.csv')
    require_columns(frames_raw, REQUIRED_FRAMES, 'keyframe.csv')
    videos = {r['video_id'].strip(): r['video_url'].strip() for r in videos_raw if r.get('video_id','').strip() and r.get('video_url','').strip()}
    shots: dict[str,list[dict[str,Any]]] = {}
    used_indexes: set[tuple[str,int]] = set()
    for row_no, raw in enumerate(shots_raw, 2):
        try:
            row = {**raw, 'shot_index': integer(raw,'shot_index'), 'start_ms': integer(raw,'start_ms'), 'end_ms': integer(raw,'end_ms'), 'start_frame_idx': integer(raw,'start_frame_idx'), 'end_frame_idx': integer(raw,'end_frame_idx')}
            video_id = row['video_id'].strip(); key = (video_id,row['shot_index'])
            if video_id not in videos: raise ValueError('video_id has no URL')
            if key in used_indexes: raise ValueError('duplicate shot_index within video')
            if min(row['start_ms'],row['end_ms'],row['start_frame_idx'],row['end_frame_idx']) < 0 or row['end_frame_idx'] < row['start_frame_idx']: raise ValueError('invalid frame/time bounds')
            used_indexes.add(key); shots.setdefault(video_id,[]).append(row)
        except Exception as exc: error('validate_shot', str(exc), row_no=row_no, row=raw)
    for values in shots.values(): values.sort(key=lambda r: r['shot_index'])
    frames, frame_ids = [], set()
    for row_no, raw in enumerate(frames_raw, 2):
        try:
            if raw['frame_id'] in frame_ids: raise ValueError('duplicate frame_id')
            fps, frame_idx = decimal(raw,'fps'), integer(raw,'frame_idx')
            if fps <= 0 or frame_idx < 0: raise ValueError('fps must be > 0 and frame_idx non-negative')
            frame_ids.add(raw['frame_id']); frames.append({**raw, 'fps':fps, 'frame_idx':frame_idx})
        except Exception as exc: error('validate_frame', str(exc), row_no=row_no, row=raw)
    return videos, shots, frames


In [ ]:
# Video probing, sampling, decoding, image features and SigLIP encoder.
def probe_video(video: Path) -> tuple[float, int, int]:
    data = json.loads(run(['ffprobe','-v','error','-select_streams','v:0','-show_entries','stream=avg_frame_rate,width,height','-of','json',str(video)]).stdout)['streams'][0]
    num, den = map(int, data['avg_frame_rate'].split('/')); fps = num / den
    if fps <= 0: raise ValueError('ffprobe returned non-positive fps')
    return fps, int(data['width']), int(data['height'])
def decode_frame(video: Path, frame_idx: int, fps: float) -> Image.Image:
    process = subprocess.run(['ffmpeg','-v','error','-ss',f'{frame_idx/fps:.9f}','-i',str(video),'-frames:v','1','-f','image2pipe','-vcodec','mjpeg','pipe:1'], check=True, capture_output=True)
    return Image.open(io.BytesIO(process.stdout)).convert('RGB').copy()
def candidate_indices(shot: dict[str,Any], fps: float, existing: set[int]) -> list[int]:
    start,end = shot['start_frame_idx'],shot['end_frame_idx']
    inner_start,inner_end = start,end
    if end-start+1 > 2*TRANSITION_MARGIN_FRAMES+1: inner_start += TRANSITION_MARGIN_FRAMES; inner_end -= TRANSITION_MARGIN_FRAMES
    step=max(1,round(fps)); values=list(range(inner_start,inner_end+1,step)); center=(inner_start+inner_end)//2
    if center not in values: values.append(center)
    values=sorted(x for x in set(values) if all(abs(x-y)>MIN_FRAME_GAP for y in existing))
    return even(values,MAX_CANDIDATES) if len(values)>MAX_CANDIDATES else values
def target_additional(shot: dict[str,Any], existing_in_shot: set[int]) -> int:
    duration=shot['end_ms']-shot['start_ms']; target=1 if duration<TARGET_INTERVAL_MS else 1+duration//TARGET_INTERVAL_MS
    return max(0, min(MAX_ADDITIONAL_PER_SHOT, target-len(existing_in_shot)))
def hsv_feature(image: Image.Image) -> np.ndarray:
    hsv=np.asarray(image.convert('HSV'),dtype=np.uint8); h=hsv[...,0].astype(np.int16)*8//256; s=hsv[...,1].astype(np.int16)*8//256; v=hsv[...,2].astype(np.int16)*8//256
    bins=(h*64+s*8+v).ravel(); hist=np.bincount(bins,minlength=512).astype(np.float32); return hist/(np.linalg.norm(hist) or 1)
def cosine(a: np.ndarray,b: np.ndarray) -> float: return float(np.dot(a,b)/(np.linalg.norm(a)*np.linalg.norm(b) or 1))

class SiglipEncoder:
    def __init__(self, enabled: bool=True):
        self.model = self.processor = None
        self.batch_size = IMAGE_BATCH_SIZE
        if not enabled: return
        from transformers import AutoModel, AutoProcessor
        self.processor = AutoProcessor.from_pretrained(SIGLIP_MODEL_ID)
        self.model = AutoModel.from_pretrained(SIGLIP_MODEL_ID).to(DEVICE)
        self.model.eval()
    def encode(self, images: list[Image.Image]) -> np.ndarray:
        if self.model is None: raise RuntimeError('SigLIP encoder disabled')
        vectors=[]; size=self.batch_size; start=0
        while start < len(images):
            batch=images[start:start+size]
            try:
                with torch.inference_mode():
                    inputs=self.processor(images=batch,return_tensors='pt')
                    inputs={key:value.to(DEVICE, non_blocking=True) for key,value in inputs.items()}
                    dtype=torch.bfloat16 if DEVICE == 'cuda' and torch.cuda.is_bf16_supported() else torch.float16
                    with torch.autocast(device_type='cuda',dtype=dtype,enabled=DEVICE == 'cuda'):
                        value=self.model.get_image_features(**inputs)
                        if not isinstance(value,torch.Tensor):
                            if getattr(value,'pooler_output',None) is not None: value=value.pooler_output
                            elif getattr(value,'image_embeds',None) is not None: value=value.image_embeds
                            else: value=value[0]
                    vectors.append(torch.nn.functional.normalize(value.float(),p=2,dim=1).cpu().numpy().astype(np.float32))
                start += len(batch)
            except RuntimeError as exc:
                if DEVICE != 'cuda' or 'out of memory' not in str(exc).lower() or size <= 1: raise
                size//=2; self.batch_size=size
                torch.cuda.empty_cache()  # OOM retry only.
        return np.concatenate(vectors,axis=0)


In [ ]:
# Deterministic hybrid selection. A successful empty result is intentionally not time-sampled.
def facility(candidates: list[int], vectors: np.ndarray, references: np.ndarray | None, quota: int) -> list[int]:
    if not candidates or quota <= 0: return []
    sim=np.clip(vectors@vectors.T,0,1); covered=np.zeros(len(candidates),np.float32) if references is None or not len(references) else np.clip(vectors@references.T,0,1).max(axis=1)
    selected=[]; available=set(range(len(candidates)))
    for _ in range(min(quota,len(candidates))):
        row=max(available,key=lambda i:(float(np.maximum(covered,sim[:,i]).sum()-covered.sum()),-candidates[i]))
        gain=float(np.maximum(covered,sim[:,row]).mean()-covered.mean())
        if gain < .01: break
        selected.append(row); available.remove(row); covered=np.maximum(covered,sim[:,row])
    return sorted(candidates[i] for i in selected)
def time_sample(shot: dict[str,Any], quota: int, existing: set[int]) -> list[int]:
    allowed=[i for i in range(shot['start_frame_idx'],shot['end_frame_idx']+1) if all(abs(i-x)>MIN_FRAME_GAP for x in existing)]
    return even(allowed,quota) if allowed else []
def hybrid_select(video: Path, fps: float, shot: dict[str,Any], existing: set[int], encoder: SiglipEncoder) -> tuple[list[int],str]:
    in_shot={i for i in existing if shot['start_frame_idx']<=i<=shot['end_frame_idx']}; quota=target_additional(shot,in_shot)
    if quota == 0: return [], 'existing_target_coverage'
    candidates=candidate_indices(shot,fps,existing)
    if not candidates: return [], 'no_candidate'
    try:
        refs=even(sorted(in_shot),MAX_REFERENCES); all_idx=candidates+refs; images=[decode_frame(video,i,fps) for i in all_idx]
        candidate_images=images[:len(candidates)]; candidate_hsv=[hsv_feature(x) for x in candidate_images]
        keep=[i for i in range(len(candidates)) if sum(candidate_hsv[i]>0)>=10]
        candidates=[candidates[i] for i in keep]; candidate_images=[candidate_images[i] for i in keep]; candidate_hsv=[candidate_hsv[i] for i in keep]
        if not candidates: return [], 'low_information'
        vectors=encoder.encode(candidate_images); refs_vec=encoder.encode(images[len(all_idx)-len(refs):]) if refs else None
        pre=facility(candidates,vectors,refs_vec,quota)
        selected=[]
        for idx in pre:
            pos=candidates.index(idx)
            if any(cosine(candidate_hsv[pos],candidate_hsv[candidates.index(old)])>.8 and float(vectors[pos]@vectors[candidates.index(old)])>.95 for old in selected): continue
            selected.append(idx)
        return selected, 'hybrid'
    except Exception as exc:
        error('hybrid', str(exc), video_id=shot['video_id'], shot_id=shot['shot_id'], traceback=traceback.format_exc())
        return time_sample(shot,quota,existing), 'fallback_time'
def cross_shot(rows: list[dict[str,Any]]) -> list[dict[str,Any]]:
    # Selection is already strongly deduped inside shots. Exact SigLIP/HSV boundary comparison is performed before export when images exist.
    return rows


In [ ]:
# R2, atomic checkpoint, SQL and export/upload.
class R2Uploader:
    def __init__(self, client: Any, bucket: str): self.client,self.bucket=client,bucket
    def upload(self, path: Path, key: str) -> dict[str,Any]:
        size=path.stat().st_size
        for attempt in range(4):
            try:
                try:
                    head=self.client.head_object(Bucket=self.bucket,Key=key)
                    if int(head.get('ContentLength',-1)) == size: return {'status':'already_present','size':size}
                except Exception: pass
                self.client.upload_file(str(path),self.bucket,key,ExtraArgs={'ContentType':'image/jpeg'})
                head=self.client.head_object(Bucket=self.bucket,Key=key)
                if int(head.get('ContentLength',-1)) != size: raise RuntimeError('R2 size mismatch after upload')
                return {'status':'uploaded','size':size}
            except Exception:
                if attempt == 3: raise
                time.sleep(2**attempt)
def r2_from_environment() -> R2Uploader:
    if not all([R2_ENDPOINT_URL,R2_BUCKET,R2_ACCESS_KEY_ID,R2_SECRET_ACCESS_KEY]): raise RuntimeError('R2 secrets/config are required for a real run')
    import boto3
    return R2Uploader(boto3.client('s3',endpoint_url=R2_ENDPOINT_URL,aws_access_key_id=R2_ACCESS_KEY_ID,aws_secret_access_key=R2_SECRET_ACCESS_KEY,region_name='auto'),R2_BUCKET)
def checkpoint_path() -> Path: return WORK_DIR/'checkpoint.json'
def save_checkpoint(state: dict[str,Any]) -> None:
    temp=checkpoint_path().with_suffix('.tmp'); temp.write_text(json.dumps(state,ensure_ascii=False,sort_keys=True),encoding='utf-8'); temp.replace(checkpoint_path())
def sql_row(row: dict[str,Any]) -> str:
    cols=['frame_id','n','video_id','shot_id','pts_time','timestamp_ms','fps','frame_idx','source','frame_path','width','height']
    return 'INSERT INTO frame\n  ('+', '.join(cols)+')\nVALUES ('+', '.join(qsql(row.get(c)) for c in cols)+')\nON CONFLICT (frame_id) DO NOTHING;\n'
def export_one(video: Path, frame_idx: int, fps: float, dest: Path) -> tuple[int,int]:
    image=decode_frame(video,frame_idx,fps); image.save(dest,'JPEG',quality=95); return image.size


In [ ]:
# Main pipeline. It processes only whole videos and does not delete a local video until upload/checkpoint succeeded.
def download(url: str, destination: Path) -> None:
    import urllib.request
    request = urllib.request.Request(url, headers={'User-Agent':'Mozilla/5.0','Accept':'video/mp4,video/*;q=0.9,*/*;q=0.8'})
    with urllib.request.urlopen(request,timeout=120) as response, destination.open('wb') as out: shutil.copyfileobj(response,out)
    if destination.stat().st_size == 0: raise RuntimeError('downloaded video is empty')
def next_sequence(video_id: str, all_ids: set[str]) -> int:
    pattern=re.compile(r'^'+re.escape(video_id)+r'_E(\d+)$'); return max([int(m.group(1)) for x in all_ids if (m:=pattern.match(x))]+[0])+1
def run_pipeline(uploader: R2Uploader, *, image_encoder_enabled: bool=True) -> dict[str,Any]:
    ERRORS_PATH.unlink(missing_ok=True); REPORT_PATH.unlink(missing_ok=True)
    videos,shots,frames=load_and_validate(); config={'interval':TARGET_INTERVAL_MS,'gap':MIN_FRAME_GAP,'model':SIGLIP_MODEL_ID,'slice':[VIDEO_START,VIDEO_END]}
    input_hash=sha256_paths([Path(VIDEO_FILE),Path(SHOT_FILE),Path(KEYFRAME_FILE)],config)
    state={'input_hash':input_hash,'completed':[],'rows':[]}
    if checkpoint_path().exists():
        state=json.loads(checkpoint_path().read_text())
        if state['input_hash'] != input_hash and not ALLOW_CHECKPOINT_MISMATCH: raise RuntimeError('checkpoint input/config hash mismatch; set ALLOW_CHECKPOINT_MISMATCH=1 only after review')
    ordered=sorted(shots); selected_videos=ordered[VIDEO_START:VIDEO_END]
    all_ids={r['frame_id'] for r in frames}|{r['frame_id'] for r in state['rows']}
    existing_by_video: dict[str,set[int]]={}
    for row in frames: existing_by_video.setdefault(row['video_id'],set()).add(row['frame_idx'])
    encoder=SiglipEncoder(enabled=image_encoder_enabled); timings={'download':0.,'selection':0.,'export':0.,'upload':0.}; fallback_count=0
    for video_id in selected_videos:
        if video_id in state['completed']: continue
        temp=WORK_DIR/f'{video_id}.mp4'; start=time.perf_counter()
        try:
            download(videos[video_id],temp); timings['download']+=time.perf_counter()-start; fps,_,_=probe_video(temp)
            chosen=[]; existing=existing_by_video.get(video_id,set()).copy()
            for shot in shots[video_id]:
                tick=time.perf_counter(); indices,reason=hybrid_select(temp,fps,shot,existing,encoder); timings['selection']+=time.perf_counter()-tick
                fallback_count += reason == 'fallback_time'
                for index in indices: chosen.append((shot,index,reason)); existing.add(index)
                jsonl(REPORT_PATH,{'video_id':video_id,'shot_id':shot['shot_id'],'selected_indices':indices,'reason':reason})
            # IDs are allocated only after selection, monotonically within the whole video.
            seq=next_sequence(video_id,all_ids); new_rows=[]
            for shot,index,reason in sorted(chosen,key=lambda x:(x[0]['shot_index'],x[1])):
                frame_id=f'{video_id}_E{seq:03d}'; seq+=1; all_ids.add(frame_id); jpg=WORK_DIR/f'{frame_id}.jpg'; tick=time.perf_counter(); width,height=export_one(temp,index,fps,jpg); timings['export']+=time.perf_counter()-tick
                key=f'{R2_KEY_PREFIX}/{video_id}/{frame_id}.jpg'; tick=time.perf_counter(); result=uploader.upload(jpg,key); timings['upload']+=time.perf_counter()-tick
                row={'frame_id':frame_id,'n':index,'video_id':video_id,'shot_id':shot['shot_id'],'pts_time':index/fps,'timestamp_ms':round(index/fps*1000),'fps':fps,'frame_idx':index,'source':'extracted','frame_path':key,'width':width,'height':height}
                new_rows.append(row); jsonl(REPORT_PATH,{**row,'reason':reason,'remote_key':key,'upload_result':result})
                jpg.unlink(missing_ok=True)
            state['rows'].extend(new_rows); state['completed'].append(video_id); save_checkpoint(state); temp.unlink(missing_ok=True)
        except Exception as exc:
            error('video',str(exc),video_id=video_id,traceback=traceback.format_exc())
    rows=sorted(state['rows'],key=lambda r:(r['video_id'], next(s['shot_index'] for ss in shots.values() for s in ss if s['shot_id']==r['shot_id']),r['frame_idx']))
    (WORK_DIR/'insert_keyframes.sql').write_text('\n'.join(sql_row(row) for row in rows),encoding='utf-8')
    summary={'videos_requested':len(selected_videos),'videos_completed':len(state['completed']),'shots':sum(len(shots[v]) for v in selected_videos),'official_frames':sum(r['source']=='official' for r in frames),'selected_uploaded':len(rows),'fallback_count':fallback_count,'timings_seconds':timings,'device':DEVICE,'gpu_name':GPU_NAME,'image_batch_size_used':encoder.batch_size,'peak_vram_bytes':torch.cuda.max_memory_allocated() if DEVICE=='cuda' else None}
    (WORK_DIR/'summary.json').write_text(json.dumps(summary,indent=2),encoding='utf-8'); return summary


In [ ]:
# Real run — uncomment only after setting input paths and Kaggle Secrets.
# summary = run_pipeline(r2_from_environment())
# print(json.dumps(summary, indent=2))


In [ ]:
# Mandatory mock test: no cloud R2 or BTC data required. This uses time fallback by disabling SigLIP.
class MockR2:
    def __init__(self): self.data={}
    def upload_file(self,path,Bucket,Key,ExtraArgs): self.data[(Bucket,Key)]=Path(path).read_bytes()
    def head_object(self,Bucket,Key):
        if (Bucket,Key) not in self.data: raise KeyError(Key)
        return {'ContentLength':len(self.data[(Bucket,Key)])}

def mock_test() -> None:
    global INPUT_DIR,WORK_DIR,VIDEO_FILE,SHOT_FILE,KEYFRAME_FILE,VIDEO_START,VIDEO_END,ALLOW_CHECKPOINT_MISMATCH
    root=Path('/tmp/kf_mock'); shutil.rmtree(root,ignore_errors=True); root.mkdir(); INPUT_DIR=WORK_DIR=root/'work'; WORK_DIR.mkdir(); video=root/'tiny.mp4'
    run(['ffmpeg','-y','-f','lavfi','-i','color=c=red:s=64x64:d=2','-f','lavfi','-i','testsrc2=s=64x64:d=2','-f','lavfi','-i','color=c=blue:s=64x64:d=2','-filter_complex','[0:v][1:v][2:v]concat=n=3:v=1:a=0','-r','10',str(video)])
    (root/'videos.csv').write_text(f'video_id,video_url\nv1,file://{video}\n'); (root/'shot.csv').write_text('shot_id,video_id,shot_index,start_ms,end_ms,start_frame_idx,end_frame_idx\ns1,v1,0,0,3000,0,29\ns2,v1,1,3000,6000,30,59\n')
    (root/'keyframe.csv').write_text('frame_id,video_id,shot_id,timestamp_ms,fps,frame_idx,source,n,pts_time,frame_path,width,height\nv1_F001,v1,,0,10,0,official,0,0,x,64,64\nv1_E001,v1,,100,10,1,extracted,1,0.1,x,64,64\n')
    VIDEO_FILE=str(root/'videos.csv'); SHOT_FILE=str(root/'shot.csv'); KEYFRAME_FILE=str(root/'keyframe.csv'); VIDEO_START=0; VIDEO_END=None; ALLOW_CHECKPOINT_MISMATCH=False
    client=MockR2(); first=run_pipeline(R2Uploader(client,'mock'),image_encoder_enabled=False); rows=json.loads(checkpoint_path().read_text())['rows']; second=run_pipeline(R2Uploader(client,'mock'),image_encoder_enabled=False)
    assert len({r['frame_idx'] for r in rows})==len(rows) and all(r['source']=='extracted' for r in rows)
    assert all(sum(r['shot_id']==s for r in rows)<=5 for s in ('s1','s2'))
    assert all(not r['frame_id'].endswith('_E001') for r in rows) and all(r['frame_path'].startswith('data2/keyframes/v1/') for r in rows)
    assert all(('mock',r['frame_path']) in client.data for r in rows) and first['official_frames']==1 and second['selected_uploaded']==len(rows)
    assert (WORK_DIR/'insert_keyframes.sql').read_text().count("source") == len(rows)
    print('mock test passed', first)
if os.environ.get('KF_RUN_MOCK_TEST', '0') == '1': mock_test()


## Kaggle production runtime

Các cell dưới đây là đường chạy thật. Chúng nạp Kaggle Secrets hoặc `.env`, dùng frame-index exact extraction, SigLIP2 cố định trên GPU và R2 upload song song.


In [ ]:
# Production configuration: Kaggle Secrets take precedence; `.env` is supported for an attached private dataset.
import importlib.util, sys

def load_dotenv_file(path: Path) -> None:
    if not path.is_file(): return
    for raw in path.read_text(encoding='utf-8').splitlines():
        line = raw.strip()
        if not line or line.startswith('#') or '=' not in line: continue
        key, value = line.split('=', 1)
        os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))

def load_kaggle_secrets() -> None:
    try:
        from kaggle_secrets import UserSecretsClient
        client = UserSecretsClient()
        for name in ('R2_ENDPOINT_URL','R2_BUCKET','R2_ACCESS_KEY_ID','R2_SECRET_ACCESS_KEY','R2_KEY_PREFIX','KF_RUN_REAL','SIGLIP_MODEL_ID','SIGLIP_MODEL_REVISION','IMAGE_BATCH_SIZE','VIDEO_START','VIDEO_END'):
            if not os.environ.get(name):
                try: os.environ[name] = client.get_secret(name)
                except Exception: pass
    except ImportError:
        pass

INPUT_DIR = Path(os.environ.get('KF_INPUT_DIR', '/kaggle/working/btc-keyframe-export'))
load_dotenv_file(Path(os.environ.get('KF_ENV_FILE', str(INPUT_DIR / '.env'))))
load_kaggle_secrets()
WORK_DIR = Path(os.environ.get('KF_WORK_DIR', '/kaggle/working/keyframe_extractor'))
WORK_DIR.mkdir(parents=True, exist_ok=True)
ERRORS_PATH, REPORT_PATH = WORK_DIR / 'errors.jsonl', WORK_DIR / 'report.jsonl'
VIDEO_FILE = os.environ.get('KF_VIDEOS_FILE', str(INPUT_DIR / 'videos.csv'))
SHOT_FILE = os.environ.get('KF_SHOT_FILE', str(INPUT_DIR / 'shot.csv'))
KEYFRAME_FILE = os.environ.get('KF_KEYFRAME_FILE', str(INPUT_DIR / 'keyframe.csv'))
VIDEO_START = int(os.environ.get('VIDEO_START', '0'))
VIDEO_END = int(os.environ['VIDEO_END']) if os.environ.get('VIDEO_END') else 1
ALLOW_CHECKPOINT_MISMATCH = os.environ.get('ALLOW_CHECKPOINT_MISMATCH', '0') == '1'
RUN_REAL = os.environ.get('KF_RUN_REAL', '1') == '1'
REQUIRE_CUDA = os.environ.get('REQUIRE_CUDA', '1') == '1'
IMAGE_ENCODER_BACKEND = 'transformers_siglip'
IMAGE_MODEL_ID = os.environ.get('SIGLIP_MODEL_ID', 'google/siglip2-base-patch16-224')
IMAGE_MODEL_REVISION = os.environ.get('SIGLIP_MODEL_REVISION') or None
IMAGE_BATCH_SIZE = int(os.environ.get('IMAGE_BATCH_SIZE', '128'))
R2_ENDPOINT_URL = os.environ.get('R2_ENDPOINT_URL')
R2_BUCKET = os.environ.get('R2_BUCKET')
R2_ACCESS_KEY_ID = os.environ.get('R2_ACCESS_KEY_ID')
R2_SECRET_ACCESS_KEY = os.environ.get('R2_SECRET_ACCESS_KEY')
R2_KEY_PREFIX = os.environ.get('R2_KEY_PREFIX', 'data2/keyframes').strip('/')
OUTPUT_ONLY = not all([R2_ENDPOINT_URL, R2_BUCKET, R2_ACCESS_KEY_ID, R2_SECRET_ACCESS_KEY])

required_packages = {'boto3': 'boto3', 'transformers': 'transformers', 'av': 'av'}
missing = [package for module, package in required_packages.items() if importlib.util.find_spec(module) is None]
if missing: subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])

if RUN_REAL and REQUIRE_CUDA and not torch.cuda.is_available():
    raise RuntimeError('GPU is required: enable a Kaggle GPU or set REQUIRE_CUDA=0 explicitly.')
print({'run_real': RUN_REAL, 'output_only': OUTPUT_ONLY, 'device': DEVICE, 'encoder_backend': IMAGE_ENCODER_BACKEND, 'model': IMAGE_MODEL_ID, 'revision': IMAGE_MODEL_REVISION, 'input_dir': str(INPUT_DIR)})


In [ ]:
# Exact frame-index decode/export and swappable GPU image encoders.
def decode_frames_exact(video: Path, frame_indices: list[int]) -> dict[int, Image.Image]:
    import tempfile
    indices = sorted(set(frame_indices))
    if not indices: return {}
    select = '+'.join(f'eq(n\\,{index})' for index in indices)
    with tempfile.TemporaryDirectory(prefix='kf_decode_') as directory:
        pattern = str(Path(directory) / 'frame_%04d.jpg')
        run(['ffmpeg','-v','error','-y','-i',str(video),'-vf',f'select={select}','-vsync','vfr',pattern])
        paths = sorted(Path(directory).glob('frame_*.jpg'))
        if len(paths) != len(indices): raise RuntimeError(f'exact decode mismatch: requested={len(indices)} got={len(paths)}')
        return {index: Image.open(path).convert('RGB').copy() for index, path in zip(indices, paths)}

class SequentialVideoDecoder:
    # Decode requested frame numbers in one forward PyAV pass with bounded RAM.
    def __init__(self, video: Path):
        self.frame_number = -1; self.container = self.stream = self.iterator = None
        try:
            import av
            self.container = av.open(str(video)); self.stream = self.container.streams.video[0]
            self.iterator = self.container.decode(self.stream)
        except Exception:
            self.close()
    def decode(self, indices: list[int], video: Path) -> dict[int, Image.Image]:
        wanted = sorted(set(indices))
        if not wanted: return {}
        if self.iterator is None or wanted[0] <= self.frame_number:
            return decode_frames_exact(video,wanted)
        result={}; wanted_set=set(wanted); last=wanted[-1]
        for frame in self.iterator:
            self.frame_number += 1
            if self.frame_number in wanted_set:
                result[self.frame_number] = Image.fromarray(frame.to_ndarray(format='rgb24')).convert('RGB').resize((224,224),Image.Resampling.BILINEAR)
            if self.frame_number >= last: break
        if len(result) != len(wanted): raise RuntimeError(f'PyAV decode mismatch: requested={len(wanted)} got={len(result)}')
        return result
    def close(self) -> None:
        if self.container is not None: self.container.close()
        self.container = self.stream = self.iterator = None

def export_frames_exact(video: Path, records: list[dict[str,Any]], stage_dir: Path) -> None:
    stage_dir.mkdir(parents=True, exist_ok=True)
    for start in range(0, len(records), FFMPEG_EXPORT_CHUNK_SIZE):
        chunk = records[start:start+FFMPEG_EXPORT_CHUNK_SIZE]
        decoded = decode_frames_exact(video, [row['frame_idx'] for row in chunk])
        for row in chunk:
            image = decoded[row['frame_idx']]
            path = stage_dir / f"{row['frame_id']}.jpg"
            image.save(path, 'JPEG', quality=95, optimize=True)
            row['local_path'], row['width'], row['height'] = path, image.width, image.height

class ImageEncoder:
    def __init__(self, enabled: bool = True):
        self.enabled, self.batch_size = enabled, IMAGE_BATCH_SIZE
        self.model = self.processor = None
        if not enabled: return
        from transformers import AutoModel, AutoProcessor
        self.processor = AutoProcessor.from_pretrained(IMAGE_MODEL_ID)
        self.model = AutoModel.from_pretrained(IMAGE_MODEL_ID).to(DEVICE)
        self.model.eval()
    def encode(self, images: list[Image.Image]) -> np.ndarray:
        if not self.enabled or self.model is None: raise RuntimeError('image encoder disabled')
        vectors=[]; size=self.batch_size; start=0
        while start < len(images):
            batch=images[start:start+size]
            while True:
                try:
                    with torch.inference_mode():
                        inputs=self.processor(images=batch,return_tensors='pt')
                        inputs={key:value.to(DEVICE, non_blocking=True) for key,value in inputs.items()}
                        autocast_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
                        with torch.autocast(device_type='cuda', dtype=autocast_dtype, enabled=DEVICE == 'cuda'):
                            value=self.model.get_image_features(**inputs)
                            if not isinstance(value,torch.Tensor):
                                if getattr(value,'pooler_output',None) is not None: value=value.pooler_output
                                elif getattr(value,'image_embeds',None) is not None: value=value.image_embeds
                                else: value=value[0]
                        value=torch.nn.functional.normalize(value.float(),p=2,dim=1)
                    vectors.append(value.detach().cpu().numpy().astype(np.float32)); start += len(batch); break
                except RuntimeError as exc:
                    if DEVICE != 'cuda' or 'out of memory' not in str(exc).lower() or size <= 1: raise
                    size//=2; self.batch_size=size; batch=images[start:start+size]; torch.cuda.empty_cache()
        return np.concatenate(vectors,axis=0)


In [ ]:
# Hybrid selection and real cross-shot boundary dedupe.
def hybrid_select_exact(video: Path, fps: float, shot: dict[str,Any], existing: set[int], encoder: ImageEncoder, decoder: SequentialVideoDecoder) -> tuple[list[int],str]:
    in_shot={index for index in existing if shot['start_frame_idx'] <= index <= shot['end_frame_idx']}
    candidates=candidate_indices(shot,fps,existing)
    if not candidates: return [], 'no_candidate'
    try:
        references=even(sorted(in_shot),MAX_REFERENCES)
        decoded=decoder.decode(candidates+references,video)
        candidate_images=[decoded[index] for index in candidates]
        histograms=[hsv_feature(image) for image in candidate_images]
        keep=[position for position,histogram in enumerate(histograms) if int(np.count_nonzero(histogram)) >= 10]
        candidates=[candidates[position] for position in keep]; candidate_images=[candidate_images[position] for position in keep]; histograms=[histograms[position] for position in keep]
        if not candidates: return [], 'low_information'
        vectors=encoder.encode(candidate_images)
        reference_vectors=encoder.encode([decoded[index] for index in references]) if references else None
        selected=facility(candidates,vectors,reference_vectors,MAX_ADDITIONAL_PER_SHOT)
        final=[]
        for index in selected:
            position=candidates.index(index)
            duplicate=any(cosine(histograms[position],histograms[candidates.index(previous)]) > .8 and float(vectors[position] @ vectors[candidates.index(previous)]) > .95 for previous in final)
            if not duplicate: final.append(index)
        return final, 'hybrid'
    except Exception as exc:
        error('hybrid',str(exc),video_id=shot['video_id'],shot_id=shot['shot_id'],traceback=traceback.format_exc())
        quota=target_additional(shot,in_shot)
        return time_sample(shot,quota,existing), 'fallback_time'

def cross_shot_dedupe_exact(video: Path, selected: list[dict[str,Any]], encoder: ImageEncoder) -> list[dict[str,Any]]:
    if not encoder.enabled: return selected
    for left,right in zip(selected,selected[1:]):
        if not left['indices'] or not right['indices']: continue
        left_index,right_index=left['indices'][-1],right['indices'][0]
        if right_index-left_index > 150: continue
        decoded=decode_frames_exact(video,[left_index,right_index])
        left_hsv,right_hsv=hsv_feature(decoded[left_index]),hsv_feature(decoded[right_index])
        vectors=encoder.encode([decoded[left_index],decoded[right_index]])
        if cosine(left_hsv,right_hsv) > .75 and float(vectors[0] @ vectors[1]) > .90 and len(right['indices']) > 1:
            right['indices'].pop(0)
            right['reason'] += '+cross_shot_dedup'
    return selected

def select_video_batched(video: Path, fps: float, video_shots: list[dict[str,Any]], existing: set[int], encoder: ImageEncoder) -> tuple[list[dict[str,Any]],int]:
    # Plan every shot first, decode the video once, then encode every unique image in large GPU batches.
    plans=[]; all_indices=set()
    for shot in video_shots:
        in_shot={index for index in existing if shot['start_frame_idx'] <= index <= shot['end_frame_idx']}
        candidates=candidate_indices(shot,fps,existing); references=even(sorted(in_shot),MAX_REFERENCES)
        plans.append({'shot':shot,'candidates':candidates,'references':references,'in_shot':in_shot})
        all_indices.update(candidates); all_indices.update(references)
    decoder=SequentialVideoDecoder(video)
    try: decoded=decoder.decode(sorted(all_indices),video)
    finally: decoder.close()
    hsv_by_index={index:hsv_feature(image) for index,image in decoded.items()}
    usable_indices=sorted(index for index,histogram in hsv_by_index.items() if int(np.count_nonzero(histogram)) >= 10)
    vectors=encoder.encode([decoded[index] for index in usable_indices]) if usable_indices else np.empty((0,1),np.float32)
    vector_by_index={index:vectors[position] for position,index in enumerate(usable_indices)}
    per_shot=[]; fallback_count=0
    for plan in plans:
        shot=plan['shot']; candidates=[index for index in plan['candidates'] if index in vector_by_index]
        references=[index for index in plan['references'] if index in vector_by_index]
        try:
            if not plan['candidates']: per_shot.append({'shot':shot,'indices':[],'reason':'no_candidate'}); continue
            if not candidates: raise RuntimeError('low_information')
            candidate_vectors=np.stack([vector_by_index[index] for index in candidates])
            reference_vectors=np.stack([vector_by_index[index] for index in references]) if references else None
            selected=facility(candidates,candidate_vectors,reference_vectors,MAX_ADDITIONAL_PER_SHOT); final=[]
            for index in selected:
                duplicate=any(cosine(hsv_by_index[index],hsv_by_index[previous]) > .8 and float(vector_by_index[index] @ vector_by_index[previous]) > .95 for previous in final)
                if not duplicate: final.append(index)
            per_shot.append({'shot':shot,'indices':final,'reason':'hybrid_batched'})
        except Exception as exc:
            error('hybrid',str(exc),video_id=shot['video_id'],shot_id=shot['shot_id'],traceback=traceback.format_exc())
            quota=target_additional(shot,plan['in_shot']); sampled=time_sample(shot,quota,existing)
            per_shot.append({'shot':shot,'indices':sampled,'reason':'fallback_time'}); fallback_count+=1
    for left,right in zip(per_shot,per_shot[1:]):
        if not left['indices'] or not right['indices']: continue
        left_index,right_index=left['indices'][-1],right['indices'][0]
        if right_index-left_index <= 150 and left_index in vector_by_index and right_index in vector_by_index and cosine(hsv_by_index[left_index],hsv_by_index[right_index]) > .75 and float(vector_by_index[left_index] @ vector_by_index[right_index]) > .90 and len(right['indices']) > 1:
            right['indices'].pop(0); right['reason'] += '+cross_shot_dedup'
    return per_shot,fallback_count


In [ ]:
# Fast upload/output copy, checkpoint and production pipeline.
class LocalOutputUploader:
    def upload(self, path: Path, key: str) -> dict[str,Any]:
        target = WORK_DIR / 'frames' / Path(key).name
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(path, target)
        return {'status':'saved_to_kaggle_output','size':target.stat().st_size,'path':str(target)}

def upload_parallel(uploader: R2Uploader, records: list[dict[str,Any]]) -> tuple[list[dict[str,Any]], list[tuple[dict[str,Any],Exception]]]:
    successes, failures = [], []
    def one(row: dict[str,Any]) -> dict[str,Any]:
        row['upload_result']=uploader.upload(row['local_path'],row['frame_path']); return row
    with ThreadPoolExecutor(max_workers=UPLOAD_WORKERS) as pool:
        future_to_row={pool.submit(one,row):row for row in records}
        for future in as_completed(future_to_row):
            row=future_to_row[future]
            try: successes.append(future.result())
            except Exception as exc: failures.append((row,exc))
    return successes, failures

def run_pipeline_production(uploader: R2Uploader) -> dict[str,Any]:
    ERRORS_PATH.unlink(missing_ok=True); REPORT_PATH.unlink(missing_ok=True)
    videos,shots,frames=load_and_validate()
    config={'algorithm':'hybrid_facility_v3_batched','model_backend':IMAGE_ENCODER_BACKEND,'model_id':IMAGE_MODEL_ID,'model_revision':IMAGE_MODEL_REVISION,'slice':[VIDEO_START,VIDEO_END],'max_candidates':MAX_CANDIDATES,'max_references':MAX_REFERENCES}
    input_hash=sha256_paths([Path(VIDEO_FILE),Path(SHOT_FILE),Path(KEYFRAME_FILE)],config)
    state={'input_hash':input_hash,'completed':[],'rows':[]}
    if checkpoint_path().exists():
        state=json.loads(checkpoint_path().read_text())
        if state['input_hash'] != input_hash and not ALLOW_CHECKPOINT_MISMATCH: raise RuntimeError('checkpoint input/config hash mismatch')
    ordered_videos=sorted(shots); selected_videos=ordered_videos[VIDEO_START:VIDEO_END]
    all_ids={row['frame_id'] for row in frames}|{row['frame_id'] for row in state['rows']}
    existing_by_video: dict[str,set[int]]={}
    for row in frames+state['rows']: existing_by_video.setdefault(row['video_id'],set()).add(int(row['frame_idx']))
    shot_order={shot['shot_id']:shot['shot_index'] for values in shots.values() for shot in values}
    encoder=ImageEncoder(enabled=True); timings={name:0.0 for name in ('download','selection','export','upload')}; fallback_count=0
    if DEVICE == 'cuda': torch.cuda.reset_peak_memory_stats()
    pending_videos=[video_id for video_id in selected_videos if video_id not in state['completed']]
    # Prefetch at most DOWNLOAD_WORKERS videos. GPU inference stays in this
    # main thread while the next bounded download overlaps its computation.
    with ThreadPoolExecutor(max_workers=DOWNLOAD_WORKERS) as downloader:
      prefetch: dict[str,Any] = {}
      for position, video_id in enumerate(pending_videos):
        for queued_id in pending_videos[position:position+DOWNLOAD_WORKERS]:
            if queued_id not in prefetch:
                queued_path=WORK_DIR/f'{queued_id}.mp4'
                prefetch[queued_id]=downloader.submit(download,videos[queued_id],queued_path)
        video_path=WORK_DIR/f'{video_id}.mp4'; stage=WORK_DIR/f'{video_id}_frames'
        try:
            tick=time.perf_counter(); prefetch.pop(video_id).result(); timings['download']+=time.perf_counter()-tick
            fps,_,_=probe_video(video_path); existing=existing_by_video.get(video_id,set()).copy()
            tick=time.perf_counter(); per_shot,video_fallbacks=select_video_batched(video_path,fps,shots[video_id],existing,encoder); timings['selection']+=time.perf_counter()-tick
            fallback_count += video_fallbacks
            sequence=next_sequence(video_id,all_ids); pending=[]
            for item in per_shot:
                for index in item['indices']:
                    frame_id=f'{video_id}_E{sequence:03d}'; sequence+=1; all_ids.add(frame_id)
                    pending.append({'frame_id':frame_id,'n':index,'video_id':video_id,'shot_id':item['shot']['shot_id'],'pts_time':index/fps,'timestamp_ms':round(index/fps*1000),'fps':fps,'frame_idx':index,'source':'extracted','frame_path':f'{R2_KEY_PREFIX}/{video_id}/{frame_id}.jpg','reason':item['reason']})
            tick=time.perf_counter(); export_frames_exact(video_path,pending,stage); timings['export']+=time.perf_counter()-tick
            tick=time.perf_counter(); uploaded,failed=upload_parallel(uploader,pending); timings['upload']+=time.perf_counter()-tick
            for row in uploaded:
                row.pop('local_path',None); state['rows'].append(row); jsonl(REPORT_PATH,{**row,'remote_key':row['frame_path']})
            save_checkpoint(state)
            for row,exc in failed: error('upload',str(exc),video_id=video_id,frame_id=row['frame_id'])
            if failed: raise RuntimeError(f'{len(failed)} R2 uploads failed; checkpoint retains verified rows for resume')
            state['completed'].append(video_id); save_checkpoint(state)
            shutil.rmtree(stage,ignore_errors=True); video_path.unlink(missing_ok=True)
        except Exception as exc:
            error('video',str(exc),video_id=video_id,traceback=traceback.format_exc())
    rows=sorted(state['rows'],key=lambda row:(row['video_id'],shot_order[row['shot_id']],row['frame_idx']))
    (WORK_DIR/'insert_keyframes.sql').write_text('\n'.join(sql_row(row) for row in rows),encoding='utf-8')
    summary={'videos_requested':len(selected_videos),'videos_completed':len(state['completed']),'selected_uploaded':len(rows),'fallback_count':fallback_count,'timings_seconds':timings,'device':DEVICE,'gpu_name':GPU_NAME,'model_backend':IMAGE_ENCODER_BACKEND,'model_id':IMAGE_MODEL_ID,'model_revision':IMAGE_MODEL_REVISION,'image_batch_size_used':encoder.batch_size,'peak_vram_bytes':torch.cuda.max_memory_allocated() if DEVICE=='cuda' else None}
    (WORK_DIR/'summary.json').write_text(json.dumps(summary,indent=2),encoding='utf-8'); return summary


In [ ]:
# Real Kaggle run. Set KF_RUN_REAL=1 in `.env` or Kaggle environment; no manual uncommenting is required.
if RUN_REAL:
    uploader = LocalOutputUploader() if OUTPUT_ONLY else r2_from_environment()
    summary=run_pipeline_production(uploader)
    print(json.dumps(summary,indent=2))
else:
    print('Dry mode: set KF_RUN_REAL=1 after attaching inputs and R2 Secrets/.env.')
